# 模板：任何新实验都从这里复制

**结构只有两半**

- 第 0 格 = 环境格。Agent 写一次，之后永不改动。
- 第 1..N 格 = 业务格。Agent 写、你逐格点运行。

**复制方法**：把本文件复制成 `projects/<新项目>/01_xxx.ipynb`，
然后把第 0 格的 `PROJECT_NAME` 改成你的项目名，并在 `config/<新项目>.toml` 写一份配置。


## 第 0 格 · 环境（固定块）

In [ ]:
# ============ 第 0 格：环境（固定块，永远不要改）============
# 唯一作用：把这台 VM 变成"能干活"的状态。幂等，随便重跑。
import os, subprocess, sys
from pathlib import Path

repo = Path("/content/colab-workbench")
if not repo.exists():
    subprocess.run(["git", "clone", "-q",
        "https://github.com/Elisabethhui/colab-workbench.git",
        str(repo)], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "pull", "-q"], check=False)
sys.path.insert(0, str(repo))

from bootstrap import setup
ctx = setup("PROJECT_NAME")   # ← 只改这一行：= config/<名字>.toml


## 业务区

下面开始才是你（和 Agent）真正关心的东西。第 0 格已经准备好：

| 变量 | 含义 |
|---|---|
| `ctx.paths.repo` | 上游项目代码（`/content/rPPG-Toolbox`），随 VM 消失，下次重 clone |
| `ctx.paths.data` | Drive 上的数据根（`Colab_Projects/_data`） |
| `ctx.paths.logs` | Drive 上的日志目录 |
| `ctx.paths.outputs` | Drive 上的产物目录 |
| `ctx.paths.cache` | 本地盘缓存（`/content/_cache`），随 VM 消失 |
| `ctx.fresh_vm` | 这台机器是不是新的 |

规则：**大文件写 Drive，临时文件写 `ctx.paths.cache`，代码永远在 `ctx.paths.repo`。**


In [ ]:
# 示例业务格 1：确认环境
print("新 VM:", ctx.fresh_vm)
print("代码:", ctx.paths.repo)
!ls {ctx.paths.repo} | head


In [ ]:
# 示例业务格 2：跑点东西，产物直接落 Drive（下次换机器还在）
import json, time
out = ctx.paths.outputs / "smoke_test.json"
out.write_text(json.dumps({"at": time.strftime("%F %T"), "ok": True}, indent=2))
print("已写入", out)


In [ ]:
# 示例业务格 3：长任务记得同时写 Drive 日志
log = ctx.paths.logs / "run.log"
with log.open("a") as fh:
    fh.write("started\n")
print("日志:", log)
